# 🎬 StreamView Analytics — Pipeline de Preparación y Modelado para Tableau
### Evaluación Parcial EP1 — Visualización de Datos (Duoc UC)

Este notebook implementa el pipeline de integración, limpieza, estandarización y modelado dimensional del catálogo corporativo de **StreamView Analytics** a partir de las fuentes `netflix_movies_detailed_up_to_2025.csv` y `netflix_tv_shows_detailed_up_to_2025.csv`.

#### Objetivos de Negocio:
1. **Retención de clientes:** Identificar la frescura y relevancia del catálogo.
2. **Engagement e interacción:** Analizar patrones de popularidad y volumen de votos.
3. **Preferencias de consumo:** Evaluar demanda por género, tipo de contenido y país de origen.
4. **Calidad de experiencia:** Diseñar métricas de valoración no sesgadas (*Weighted Rating*).

#### Decisiones Arquitectónicas Implementadas:
- **Resolución de Colisiones:** Creación de `content_key` único para evitar ambigüedades en 397 `show_id` duplicados entre películas y series.
- **Restitución de Variables Estratégicas:** Conservación de `country` (92.9% poblado), `date_added` (100% poblado), `vote_count` (100% poblado) y `language`.
- **Eliminación de Redundancia:** Descarte de `rating` (duplicado idéntico al 100% de `vote_average`) y de métricas sin densidad suficiente (`budget`, `revenue`, `duration`).
- **Arquitectura de Géneros (Long Format & Bridge):** Abandono del esquema *Wide* (`genre_1`...`genre_8`) en favor de un modelo desanidado (*Long Format*) con taxonomía unificada, permitiendo filtros y agregaciones nativas en Tableau.
- **Ingeniería de Características:** Creación de `primary_country` (mapeo geográfico inmediato en Tableau), desglose temporal (`year_added`, `month_added`), flag `has_valid_votes` y rating bayesiano ponderado (`weighted_rating`).


## 1. Configuración del Entorno y Carga de Fuentes


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Configuración de visualización
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

# Rutas de entrada y salida
DATA_DIR = Path('../data')
PROCESSED_DIR = DATA_DIR / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df_movies_raw = pd.read_csv(DATA_DIR / 'netflix_movies_detailed_up_to_2025.csv')
df_tv_shows_raw = pd.read_csv(DATA_DIR / 'netflix_tv_shows_detailed_up_to_2025.csv')

print(f"Películas cargadas: {df_movies_raw.shape[0]:,} filas × {df_movies_raw.shape[1]} columnas")
print(f"Series cargadas:    {df_tv_shows_raw.shape[0]:,} filas × {df_tv_shows_raw.shape[1]} columnas")


Películas cargadas: 16,000 filas × 18 columnas
Series cargadas:    16,000 filas × 16 columnas


## 2. Limpieza Base y Resolución de Identificadores
La auditoría técnica demostró que existen **397 `show_id` duplicados** que corresponden a títulos diferentes entre películas y series. Para garantizar la integridad referencial en Tableau, se genera una clave primaria compuesta: `content_key = fuente + '_' + show_id`.


In [2]:
def limpiar_catalogo_base(df, fuente):
    """Aplica estandarización de tipos, normalización de strings y genera clave única."""
    df = df.copy()

    # Estandarización de nombres de columnas
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(' ', '_', regex=False)
    )

    # Limpieza de valores nulos o cadenas vacías en columnas de texto
    columnas_texto = df.select_dtypes(include=['object', 'string']).columns
    for col in columnas_texto:
        df[col] = df[col].astype('string').str.strip()
        df[col] = df[col].replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA})

    # Conversión temporal
    if 'date_added' in df.columns:
        df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')

    # Conversión numérica de métricas
    columnas_numericas = ['show_id', 'release_year', 'popularity', 'vote_count', 'vote_average']
    for col in columnas_numericas:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Clave primaria compuesta para evitar colisiones
    df['content_key'] = fuente + '_' + df['show_id'].astype('Int64').astype(str)

    # Deduplicación por show_id dentro de la misma fuente
    df = df.drop_duplicates(subset='show_id', keep='first').reset_index(drop=True)
    return df

df_movies_clean = limpiar_catalogo_base(df_movies_raw, 'movies')
df_tv_clean = limpiar_catalogo_base(df_tv_shows_raw, 'tv_shows')

# Columnas seleccionadas para el catálogo unificado
cols_relevantes = [
    'content_key', 'show_id', 'type', 'title', 'release_year',
    'date_added', 'country', 'language', 'genres',
    'popularity', 'vote_count', 'vote_average'
]

df_catalogo = pd.concat(
    [df_movies_clean[cols_relevantes], df_tv_clean[cols_relevantes]],
    ignore_index=True,
    sort=False
)

# Validaciones de integridad
assert len(df_catalogo) == len(df_movies_clean) + len(df_tv_clean), "Error en el número de filas"
assert df_catalogo['content_key'].is_unique, "Error: content_key debe ser estrictamente única"

print(f"Catálogo unificado: {len(df_catalogo):,} títulos únicos")
print("Distribución por tipo:")
print(df_catalogo['type'].value_counts())


Catálogo unificado: 31,991 títulos únicos
Distribución por tipo:
type
Movie      16000
TV Show    15991
Name: count, dtype: Int64


## 3. Poda Justificada de Variables
En el pipeline original se habían eliminado columnas esenciales bajo el supuesto erróneo de falta de datos.
La auditoría demostró:
- `country`: **92.9% de datos válidos** (restituida para análisis geográfico).
- `date_added`: **100% de datos válidos** (restituida para análisis temporal de incorporación de contenidos).
- `vote_count`: **100% de datos válidos** (restituida para ponderar ratings y mitigar sesgos).
- `language`: **100% de datos válidos** (restituida para segmentación lingüística).
- `rating`: **100% redundante** con `vote_average` (eliminada).
- `budget` y `revenue`: **>65% de ceros** y ausentes en TV (eliminadas).
- `duration`: Vacía en películas y constante (`1 Seasons`) en TV (eliminada).


In [3]:
# Verificación de integridad de variables restituidas
reporte_cobertura = pd.DataFrame({
    'Total_Registros': len(df_catalogo),
    'No_Nulos': df_catalogo[['country', 'date_added', 'vote_count', 'language']].notna().sum(),
    'Porcentaje_Valido': (df_catalogo[['country', 'date_added', 'vote_count', 'language']].notna().mean() * 100).round(2)
})
print(reporte_cobertura)


            Total_Registros  No_Nulos  Porcentaje_Valido
country               31991     29730             92.930
date_added            31991     31991            100.000
vote_count            31991     31991            100.000
language              31991     31991            100.000


## 4. Normalización y Armonización de Taxonomía de Géneros
Las fuentes originales utilizan vocabularios dispares para el mismo concepto (ej. películas usa `Action` y `Adventure` por separado, mientras que series usa `Action & Adventure`).

Se aplica un mapeo de taxonomía para unificar ambas fuentes antes del desanidado (*explode*).


In [4]:
TAXONOMY_MAP = {
    'Action & Adventure': ['Action', 'Adventure'],
    'Sci-Fi & Fantasy': ['Science Fiction', 'Fantasy'],
    'War & Politics': ['War', 'Politics']
}

def parse_and_harmonize_genres(genre_str):
    """Limpia y armoniza los géneros unificando taxonomías entre películas y series."""
    if pd.isna(genre_str) or not str(genre_str).strip():
        return []
    cleaned = str(genre_str).replace('[', '').replace(']', '').replace("'", '').replace('"', '')
    raw_list = [g.strip() for g in cleaned.split(',') if g.strip()]
    harmonized = []
    for g in raw_list:
        if g in TAXONOMY_MAP:
            harmonized.extend(TAXONOMY_MAP[g])
        elif g.lower() == 'unknown':
            continue
        else:
            harmonized.append(g)
    
    # Preservar unicidad dentro del mismo título
    seen = set()
    result = []
    for g in harmonized:
        if g not in seen:
            seen.add(g)
            result.append(g)
    return result

df_catalogo['genres_list'] = df_catalogo['genres'].apply(parse_and_harmonize_genres)
df_catalogo['genres_harmonized'] = df_catalogo['genres_list'].apply(lambda x: ', '.join(x) if x else 'Unknown')

# Generación de la tabla puente (Long Format) para Tableau
df_genres_bridge = (
    df_catalogo[['content_key', 'genres_list']]
    .explode('genres_list')
    .rename(columns={'genres_list': 'genre'})
)
df_genres_bridge['genre'] = df_genres_bridge['genre'].fillna('Unknown').replace('', 'Unknown')
df_genres_bridge = df_genres_bridge.drop_duplicates().reset_index(drop=True)

print(f"Total relaciones Título-Género (Long Format): {len(df_genres_bridge):,}")
print(f"Géneros únicos armonizados: {df_genres_bridge['genre'].nunique()}")
print("Top 10 géneros más frecuentes:")
print(df_genres_bridge['genre'].value_counts().head(10))


Total relaciones Título-Género (Long Format): 71,139
Géneros únicos armonizados: 26
Top 10 géneros más frecuentes:
genre
Drama              14769
Comedy              9105
Action              5227
Animation           4069
Thriller            3769
Adventure           3756
Science Fiction     3412
Fantasy             3408
Crime               3203
Family              3014
Name: count, dtype: int64


## 5. Feature Engineering: Calificación Bayesiana y Enriquecimiento
Para resolver los sesgos detectados en la auditoría (4,555 títulos en 0.0 con 0 votos y 482 títulos con 10.0 sustentados en ≤2 votos), se calcula la **calificación ponderada bayesiana (Weighted Rating)**:

$$\\text{WR} = \\frac{v}{v + m} \\cdot R + \\frac{m}{v + m} \\cdot C$$

Donde:
- $v$: número de votos (`vote_count`).
- $R$: calificación promedio (`vote_average`).
- $m$: umbral de votos (percentil 70 de títulos con votos > 0).
- $C$: calificación media global del catálogo calificado.

Adicionalmente, se extrae `primary_country` para geocodificación automática en mapas de Tableau.


In [5]:
# 1. País primario para geocodificación nativa
df_catalogo['primary_country'] = (
    df_catalogo['country']
    .dropna()
    .str.split(',')
    .str[0]
    .str.strip()
)
df_catalogo['primary_country'] = df_catalogo['primary_country'].fillna('Unknown')

# 2. Desglose temporal para análisis de cohortes
df_catalogo['year_added'] = df_catalogo['date_added'].dt.year.astype('Int64')
df_catalogo['month_added'] = df_catalogo['date_added'].dt.month.astype('Int64')

# 3. Flag de calidad de votos
df_catalogo['has_valid_votes'] = df_catalogo['vote_count'] > 0

# 4. Cálculo de Calificación Ponderada Bayesiana
voters = df_catalogo[df_catalogo['vote_count'] > 0]
C = voters['vote_average'].mean()
m = voters['vote_count'].quantile(0.70)

v = df_catalogo['vote_count']
R = df_catalogo['vote_average']

df_catalogo['weighted_rating'] = np.where(
    v > 0,
    (v / (v + m)) * R + (m / (v + m)) * C,
    np.nan
)
df_catalogo['weighted_rating'] = df_catalogo['weighted_rating'].round(2)

print(f"Parámetros bayesianos calculados: Media Global C = {C:.2f} | Umbral m = {m:.1f} votos")
print(f"Títulos sin votos aislados (has_valid_votes = False): {(~df_catalogo['has_valid_votes']).sum():,}")


Parámetros bayesianos calculados: Media Global C = 6.63 | Umbral m = 184.0 votos
Títulos sin votos aislados (has_valid_votes = False): 4,560


## 6. Generación y Exportación de Datasets para Tableau
Se exportan 3 datasets optimizados en la carpeta `data/processed/`:
1. `streamview_catalog_clean.csv`: Grano = 1 fila por título (31,991 registros). Para KPIs globales, mapas, cohortes temporales y filtros generales.
2. `streamview_genres_bridge.csv`: Grano = 1 fila por relación título-género (71,139 registros). Modelo relacional para Tableau.
3. `streamview_tableau_master.csv`: Tabla denormalizada (71,139 registros) lista para conexión directa de archivo único.


In [6]:
cols_catalogo_final = [
    'content_key', 'show_id', 'type', 'title', 'release_year',
    'date_added', 'year_added', 'month_added', 'country', 'primary_country',
    'language', 'popularity', 'vote_average', 'vote_count',
    'has_valid_votes', 'weighted_rating', 'genres_harmonized'
]

df_catalog_clean = df_catalogo[cols_catalogo_final].copy()

# Generación de tabla maestra denormalizada
df_tableau_master = df_catalog_clean.merge(df_genres_bridge, on='content_key', how='left')

# Rutas de exportación
path_catalog = PROCESSED_DIR / 'streamview_catalog_clean.csv'
path_bridge = PROCESSED_DIR / 'streamview_genres_bridge.csv'
path_master = PROCESSED_DIR / 'streamview_tableau_master.csv'

df_catalog_clean.to_csv(path_catalog, index=False, encoding='utf-8')
df_genres_bridge.to_csv(path_bridge, index=False, encoding='utf-8')
df_tableau_master.to_csv(path_master, index=False, encoding='utf-8')

print(f"✅ Exportado catálogo limpio:  {path_catalog} ({len(df_catalog_clean):,} filas)")
print(f"✅ Exportada tabla puente:     {path_bridge} ({len(df_genres_bridge):,} filas)")
print(f"✅ Exportada tabla maestra:    {path_master} ({len(df_tableau_master):,} filas)")


✅ Exportado catálogo limpio:  ..\data\processed\streamview_catalog_clean.csv (31,991 filas)
✅ Exportada tabla puente:     ..\data\processed\streamview_genres_bridge.csv (71,139 filas)
✅ Exportada tabla maestra:    ..\data\processed\streamview_tableau_master.csv (71,139 filas)


## 7. Verificación de Métricas y KPIs de Preparación


In [7]:
kpis = {
    'Total Títulos': len(df_catalog_clean),
    'Total Películas': (df_catalog_clean['type'] == 'Movie').sum(),
    'Total Series': (df_catalog_clean['type'] == 'TV Show').sum(),
    'Países Representados': df_catalog_clean['primary_country'].nunique(),
    'Géneros Armonizados': df_genres_bridge['genre'].nunique(),
    'Popularidad Mediana': df_catalog_clean['popularity'].median(),
    'Rating Promedio Ponderado': df_catalog_clean['weighted_rating'].dropna().mean(),
    'Cobertura de Votos Válidos': f"{df_catalog_clean['has_valid_votes'].mean() * 100:.1f}%"
}

df_kpis = pd.DataFrame(list(kpis.items()), columns=['KPI', 'Valor'])
df_kpis
